# Recitation 3 — Dynamic Programming & Greedy Algorithms (Java)

Save this notebook in your `cs301-notebooks` project and open it in your IDE.
Use the Java (IJava) kernel.
See [Recitation 0](https://harryc919.github.io/TA-CS301/recitation0.html) for setup.

Write your reasoning in the Markdown answer cells (double-click to edit, Shift+Enter to render).
For Exercise 2, Part B, edit and run the starter definition, then run its self-check.
An untouched starter fails the check; that is expected. Keep the signature unchanged.
The Huffman coding section at the end is optional extra content and has its own starter and check.
Reference answers are not in this notebook; they are on the
[Recitation 3 page](https://harryc919.github.io/TA-CS301/recitation3.html). Nothing is submitted or graded.

IJava accepts class definitions and statements directly, without a main method. Each starter defines a class named `Solution`: re-run the matching starter cell before its check cell. A check failure throws an exception; no JVM assertion flag is needed.


## What we are practicing

This Tuesday session covers the dynamic-programming lectures from the second half of Week 2 (Sum of Products, Matrix Chain Multiplication) and the greedy-algorithm lectures from early Week 3 (Activity Selection, Coin Change). Graph algorithms come later and are not part of this session.

- Follow one common path for a DP problem: define the subproblem, write the key math function (the recurrence), fix base cases and fill order, and recover the answer.
- Count DP cost as (number of subproblems) × (work per subproblem).
- Explain what a greedy algorithm commits to and why that commitment needs a proof.
- Use a counterexample to show greedy coin change failing, and explain why it works for some coin systems.
- Prove a greedy rule correct with an exchange argument.
- Derive the LCS recurrence and implement a table fill plus traceback that returns one longest common subsequence.

Prerequisites: recursion, 2-D arrays, the cost of sorting, and the lecture material on DP and greedy algorithms. We reuse the Θ-bound reasoning from Recitation 2.

Spend about 15 minutes on the two review sections (the three-coin test and the 1–2–5 proof can wait until after the session), then about 40 minutes on the core practice: Exercise 1 (15 minutes) and Exercise 2 (25 minutes). Leave about 10 minutes for discussion. Timing is a guide, not a completion requirement. The Huffman coding section at the end is optional extra content.

These exercises are optional and ungraded. Use hints when needed. AI tools are allowed during recitation, but you should be able to explain, verify, and modify your answer. Follow the instructor's separate assignment and exam policies.

## 1. A common path for DP problems

Dynamic programming applies when an optimal answer is built from optimal answers to smaller subproblems (*optimal substructure*) and the same subproblems come up again and again (*overlapping subproblems*). Plain recursion re-solves those shared subproblems; DP solves each one once and stores the result.

1. **Define the subproblem.** Say in words what `OPT(i, j)` means and which indices it uses. If you cannot say it in one sentence, the state is not clear yet.
2. **Write the key math function.** Express `OPT` for one state in terms of smaller states by trying every possibility for one decision, usually the first or the last one.
3. **Fix the base cases and the fill order.** Use top-down recursion with memoization, or fill a table bottom-up so that every value a cell depends on is already computed.
4. **Find and recover the answer.** Name the cell that holds the final answer. To return the solution itself, not just its value, store or retrace the choice made at each cell.
5. **Count the cost.** Time ≈ (number of states) × (work per state); space ≈ the size of the table.

### Worked anchor: Matrix Chain Multiplication

Matrix $A_{k}$ has dimensions $p_{k-1}$ × $p_{k}$. Multiplying a p × q matrix by a q × r matrix costs pqr scalar multiplications. We want the parenthesization of $A_{1}A_{2}$⋯$A_{n}$ with the fewest multiplications.

1. **Subproblem:** `m[i][j]` = minimum cost to compute $A_{i}$⋯$A_{j}$.
2. **Key math function:** the last multiplication splits the chain at some k, so `m[i][j] = min over i ≤ k < j of ( m[i][k] + m[k+1][j] + p(i−1)·p(k)·p(j) )`.
3. **Base case and order:** `m[i][i] = 0`. Fill by increasing chain length j − i, because every split uses strictly shorter chains.
4. **Answer:** `m[1][n]`. To recover the parenthesization, store the best split `s[i][j]` for each cell.
5. **Cost:** Θ(n²) states × O(n) choices of k = Θ(n³) time, and Θ(n²) space.

Example: p = (10, 30, 5, 60). Computing ($A_{1}A_{2}$)$A_{3}$ costs 10·30·5 + 10·5·60 = 4,500, while $A_{1}$($A_{2}A_{3}$) costs 30·5·60 + 10·30·60 = 27,000. So `m[1][3] = 4,500`, with split k = 2. Plain recursion tries every parenthesization, and their number grows exponentially. The DP table has only Θ(n²) distinct subproblems.

Sum of Products from lecture follows the same five steps. When you meet a new DP problem, write each step down explicitly before coding. Most DP bugs come from a vague subproblem definition or a fill order that reads cells before they are filled.

## 2. Why greedy, and when it is right

A greedy algorithm makes one locally best choice at each step and never revisits it. There is no table of subproblems: often it is just a sort followed by one scan, so it is simpler and faster than DP. The price is that a locally best choice is not automatically globally optimal. Every greedy algorithm needs a proof.

### Four ways to prove a greedy algorithm correct

1. **Exchange argument** (the most reusable). Take an optimal solution OPT and the first greedy choice g. If OPT already contains g, move on to the rest of the problem. Otherwise, find an element o of OPT that g can replace, and check two things: the new solution is still *feasible* (valid), and its value is *no worse*. So some optimal solution contains g. You will write one in Exercise 1; Huffman coding in the Extra section uses the same idea.
2. **Greedy stays ahead.** Choose a measure of progress and prove by induction that greedy is never behind OPT. In Activity Selection, if $g_{i}$ and $o_{i}$ are the i-th activities chosen by greedy and by OPT, then finish($g_{i}$) ≤ finish($o_{i}$) for every i, so greedy always has at least as much room left and picks at least as many activities. Caution: being ahead in some measure proves nothing unless that measure implies the final objective.
3. **Greedy choice + optimal substructure, by induction.** This is the overall frame. The *greedy-choice property* says some optimal solution contains the current greedy choice (usually shown with an exchange). *Optimal substructure* says that solving the remaining subproblem optimally then gives an optimal solution to the whole problem. Induction on the number of choices finishes the proof.
4. **Cut property** (used for minimum spanning trees in Week 4). For any way of splitting the vertices into two sides that no chosen edge crosses, a lightest edge between the two sides is safe to add. Kruskal's and Prim's algorithms both rely on this.

### Coin change: same algorithm, different verdicts

Coin values are 1 = $c_{1}$ < $c_{2}$ < ⋯ < $c_{n}$, with an unlimited supply of each. To pay amount N, choose counts $x_{i}$ ≥ 0 with Σ $x_{i}c_{i}$ = N that minimize the number of coins Σ $x_{i}$. Greedy repeatedly takes the largest coin that fits.

```pseudocode
GREEDY-CHANGE(coins sorted descending, amount)
    count = 0
    for each coin c in coins:
        count = count + floor(amount / c)
        amount = amount mod c
    return count            // assumes a 1-value coin exists
```

| Coins | Amount | Greedy | Optimal | Verdict |
| --- | --- | --- | --- | --- |
| `{1, 5, 10, 25}` | 63 | 25+25+10+1+1+1 (6) | 6 coins | Greedy is optimal for every amount |
| `{1, 2, 4, 8}` | 15 | 8+4+2+1 (4) | 4 coins | Optimal: each coin divides the next larger one |
| `{1, 3, 4}` | 6 | 4+1+1 (3) | 3+3 (2) | Greedy fails |
| `{1, 10, 25}` | 30 | 25+1+1+1+1+1 (6) | 10+10+10 (3) | Greedy fails |

For `{1, 3, 4}`, the optimal 3+3 cannot be exchanged for a solution that starts with the 4 without adding coins: 6 − 4 = 2 needs two 1s. The greedy-choice property fails for this coin system, so no proof can exist.

### Proving a coin system works: bound the smaller coins

One recipe proves every greedy-optimal coin system below. Let R be the total value of all coins smaller than the largest coin $c_{n}$ in an optimal solution.

1. **Exchange rules.** Each rule replaces some coins by fewer coins of the same total value, so an optimal solution can never contain the group being replaced. This gives upper bounds on the counts of the smaller coins.
2. **Bound R.** Use those bounds to show R < $c_{n}$.
3. **Match greedy.** If an optimal solution used fewer than ⌊N / $c_{n}$⌋ coins of value $c_{n}$, its smaller coins would have to add up to at least $c_{n}$, which is impossible. So it uses exactly ⌊N / $c_{n}$⌋ of them, just like greedy.
4. **Repeat** on the remaining amount with the remaining coins (induction).

**Divisibility chain.** Suppose every coin is a multiple of the previous one: $c_{i+1}$ = $k_{i}$·$c_{i}$ with integer $k_{i}$ ≥ 2, as in {1, 2, 4, 8, 16} or {1, 2, 10, 50, 100}. Exchange rule: $k_{i}$ coins of value $c_{i}$ become one coin of value $c_{i+1}$. So $x_{i}$ ≤ $k_{i}$ − 1, and R ≤ Σ ($k_{i}$ − 1)·$c_{i}$ = Σ ($c_{i+1}$ − $c_{i}$) = $c_{n}$ − 1 < $c_{n}$. The counts work like the digits of a number in a mixed base (binary for powers of two), and greedy computes exactly those digits.

**US coins `{1, 5, 10, 25}`.** Here 10 does not divide 25, so the chain argument does not apply, yet the same recipe works. Write $x_{1}$, $x_{5}$, $x_{10}$ for the counts in an optimal solution.

| Bound in any optimal solution | Exchange that rules out more |
| --- | --- |
| $x_{1}$ ≤ 4 | 1+1+1+1+1 → 5 |
| $x_{5}$ ≤ 1 | 5+5 → 10 |
| $x_{10}$ ≤ 2 | 10+10+10 → 25+5 |
| $x_{10}$ = 2 ⇒ $x_{5}$ = 0 | 10+10+5 → 25 |

If $x_{10}$ = 2, then R ≤ 20 + 0 + 4 = 24; otherwise R ≤ 10 + 5 + 4 = 19. Either way R < 25, so an optimal solution uses exactly ⌊N / 25⌋ quarters. Repeating the argument, the 1s and 5s add up to at most 9 < 10, and the 1s to at most 4 < 5. Greedy matches the optimal solution at every coin. The divisibility chain is therefore a *sufficient* condition, not a necessary one.

**Try it:** prove that greedy is optimal for the 1–2–5 system `{1, 2, 5, 10, 20, 50, 100}` used by many currencies.

### Exactly when do three coins work?

For three coins {1, a, b} with 1 < a < b, write b = q·a + r with q = ⌊b / a⌋ and 0 ≤ r < a. Then:

**Greedy is optimal ⇔ r = 0 or q + r ≥ a.** Equivalently, greedy fails exactly when 0 < r < a − q.

This is a necessary *and* sufficient condition. The failing direction comes with a counterexample. If 0 < r < a − q, take N = a + b − 1. Since N < 2b, greedy uses one b and then a − 1 ones: a coins in total. But N = (q + 1)·a + (r − 1), which uses q + r < a coins. The other direction (greedy is optimal whenever the condition holds) needs a separate argument, which we do not show here.

| Coins | q | r | Greedy optimal? | Counterexample N = a + b − 1 |
| --- | --- | --- | --- | --- |
| `{1, 2, 5}` | 2 | 1 | Yes (q + r = 3 ≥ 2) | — |
| `{1, 5, 10}` | 2 | 0 | Yes (r = 0) | — |
| `{1, 3, 4}` | 1 | 1 | No (1 < 3 − 1) | 6: 4+1+1 vs 3+3 |
| `{1, 5, 12}` | 2 | 2 | No (2 < 5 − 2) | 16: 12+1+1+1+1 vs 5+5+5+1 |

### A checklist for coin-change questions

1. Is there a divisibility chain? Then greedy is optimal.
2. Only three coins? Apply the q + r test.
3. Looking for a counterexample? Try amounts just above a large coin that several copies of a middle coin can pay, such as a + b − 1.
4. Proving correctness? Bound the smaller coins with exchange rules, show they add up to less than the next coin, and induct.
5. No proof in sight? Use DP: `C(0) = 0`, `C(a) = 1 + min over coins c ≤ a of C(a − c)`, in Θ(amount × number of coin values) time.

For general coin systems there are also known bounds on where the smallest counterexample can lie, and polynomial-time tests of whether greedy always works. They are useful for checking a system by computer, but they do not replace a proof when one is asked for.

**Takeaway:** whether greedy is correct depends on the structure of the input (here, the coin system), not on the algorithm alone.

**Quick check:** use the three-coin test on `{1, 7, 10}`. If greedy fails, give a counterexample.

<details><summary>Hint for the 1–2–5 Try it · Which exchanges do you need?</summary>

Look for small groups that equal one larger coin, or a larger coin plus a smaller one: 1+1 → 2, 2+2+2 → 5+1, 2+2+1 → 5. The pattern repeats ten times larger for 10, 20, and 50.

</details>

### Your answer — Greedy review (optional)

**Try it, 1–2–5 system:** List your exchange rules, then show the smaller coins add up to less than each coin.

**Quick check, {1, 7, 10}:** Compute q and r, apply the test, and give a counterexample if greedy fails.


## Core practice · about 40 minutes

Try each prompt before opening the reference explanation. Written answers can be short equations, small tables, or sentences. Use your IDE or the downloadable notebook to run code: the editors on this page display and copy code but do not run or grade it.

1. Exercise 1 · greedy scheduling, 1 || Lmax — 15 minutes.
2. Exercise 2 · LCS, Part A: the key math function — about 10 minutes.
3. Exercise 2 · LCS, Part B: return the LCS — about 15 minutes.

> Browser code drafts are temporary. Copy your work before leaving the page. Tab indents, Shift+Tab unindents, and Escape then Tab leaves the editor. Refresh restores templates; Reset code restores the selected language's template.

## Exercise 1 · Scheduling to minimize maximum lateness

**Trace, complexity, and a proof · 15 minutes.** One machine must process n jobs, one at a time, without interruption. All jobs are available at time 0. Job j takes $p_{j}$ > 0 time units and has due date $d_{j}$. If job j finishes at time $C_{j}$, its *lateness* is $L_{j}$ = $C_{j}$ − $d_{j}$ (negative if it finishes early). The goal is to minimize the maximum lateness $L_{\max}$ = $\max_{j}$ $L_{j}$. In scheduling notation this problem is written `1 || Lmax`.

Greedy rule **Earliest Due Date (EDD)**: run the jobs in order of increasing due date.

```pseudocode
EDD-SCHEDULE(jobs)              // job j has p[j] > 0 and due date d[j]
    order = jobs sorted by d, ascending (ties in any order)
    t = 0
    Lmax = −∞
    for each job j in order:
        t = t + p[j]            // completion time C[j]
        Lmax = max(Lmax, t − d[j])
    return (order, Lmax)
```

| Job | J1 | J2 | J3 | J4 |
| --- | --- | --- | --- | --- |
| Processing time $p_{j}$ | 4 | 2 | 1 | 3 |
| Due date $d_{j}$ | 5 | 7 | 9 | 6 |

1. **Trace.** Run EDD on the table. List the order, each completion time $C_{j}$, each lateness $L_{j}$, and $L_{\max}$. Then compute $L_{\max}$ for Shortest Processing Time first (SPT), the order of increasing $p_{j}$.
2. **Complexity.** Give the time and space of EDD-SCHEDULE for n jobs. Which step dominates?
3. **Correctness.** Prove that EDD minimizes $L_{\max}$ in three steps, listed below.

- **3.1** Suppose a schedule with no idle time is *not* in EDD order. Explain why it must contain two *adjacent* jobs i, j, with i immediately before j and $d_{i}$ > $d_{j}$ (an inversion).
- **3.2** Swap i and j. Show that no other job's lateness changes, and that the new lateness of each of i and j is at most the old $L_{j}$.
- **3.3** Conclude: starting from any optimal schedule, how do you reach an EDD schedule without making $L_{\max}$ worse?

<details><summary>Hint 1 · What does a swap change?</summary>

Let the pair start at time t. Before the swap, $C_{i}$ = t + $p_{i}$ and $C_{j}$ = t + $p_{i}$ + $p_{j}$. After it, j finishes at t + $p_{j}$ and i at t + $p_{i}$ + $p_{j}$. Jobs before and after the pair keep their completion times.

</details>

<details><summary>Hint 2 · Compare with the old lateness of j</summary>

After the swap, job i finishes exactly when j used to finish, but i has the *later* due date. Job j now finishes earlier than before. Compare both new lateness values with $L_{j}$ = t + $p_{i}$ + $p_{j}$ − $d_{j}$.

</details>

### Your answer — Exercise 1

1. **Trace:** EDD order, completion times, lateness, and $L_{\max}$; then $L_{\max}$ for SPT.
2. **Complexity:** Time and space, and the step that dominates.
3. **Correctness:**
   - 3.1 Why a non-EDD schedule has an adjacent inversion:
   - 3.2 Lateness of i and j before and after the swap:
   - 3.3 How to reach an EDD schedule without increasing $L_{\max}$:


## Exercise 2 · LCS, Part A: set up the key math function

**Written reasoning and a small table · about 10 minutes.** A *subsequence* keeps some characters of a string in their original order; they do not need to be adjacent. For example, `"ACE"` is a subsequence of `"ABCDE"`. Given strings X = $x_{1}$⋯$x_{m}$ and Y = $y_{1}$⋯$y_{n}$, a *longest common subsequence* (LCS) is a longest string that is a subsequence of both. Trying all $2^{m}$ subsequences of X is hopeless for long inputs, so we use the five-step path from Section 1.

1. **Subproblem.** Let L(i, j) be the LCS length of the prefixes X[1..i] and Y[1..j]. Which cell holds the final answer?
2. **Last characters match.** If $x_{i}$ = $y_{j}$, write L(i, j) in terms of a smaller subproblem. In one sentence, explain why it is safe to match these two characters with each other.
3. **Last characters differ.** If $x_{i}$ ≠ $y_{j}$, an LCS cannot use both of them as its last character. Which smaller subproblems remain? Write L(i, j).
4. **Base cases and fill order.** What are L(0, j) and L(i, 0)? Which neighboring cells does L(i, j) read, and in what order can you fill the table?
5. **Fill the table** for X = `"ABCB"` (rows) and Y = `"BDCAB"` (columns). Give the LCS length and one LCS.
6. **Cost.** How many cells are there, and how much work does each take? Give time and space.

<details><summary>Hint 1 · Think about the last characters</summary>

Any common subsequence of X[1..i] and Y[1..j] either uses $x_{i}$ and $y_{j}$ as a matched final pair, or ignores $x_{i}$, or ignores $y_{j}$. When $x_{i}$ ≠ $y_{j}$, the first option is impossible.

</details>

<details><summary>Hint 2 · The table shape</summary>

Use rows i = 0..4 for "", A, B, C, B and columns j = 0..5 for "", B, D, C, A, B. Row 0 and column 0 are all zeros. Each cell reads only the cell above, the cell to its left, and the diagonal cell above-left.

</details>

### Your answer — Exercise 2, Part A

1. **Meaning of L(i, j) and where the answer is:** Write your answer here.
2. **Case $x_i = y_j$:** L(i, j) = … and why matching is safe.
3. **Case $x_i \neq y_j$:** L(i, j) = …
4. **Base cases and fill order:** Write your answer here.
5. **Table for "ABCB" / "BDCAB":** Fill the table below, then give the LCS length and one LCS.

| | "" | B | D | C | A | B |
|---|---|---|---|---|---|---|
| "" | | | | | | |
| A | | | | | | |
| B | | | | | | |
| C | | | | | | |
| B | | | | | | |

6. **Cost:** Number of cells, work per cell, time and space.


## Exercise 2 · LCS, Part B: return the LCS

**Coding and explanation · about 15 minutes.** Implement the function from Part A. Return *one* longest common subsequence of `a` and `b` as a string, not just its length. The starter allocates the table and provides the loops. Fill in the key math function, then write the traceback.

- Each input has 0–1,000 ASCII characters. Comparison is case-sensitive: `'a' ≠ 'A'`.
- Several longest answers may exist; any one is accepted. The checks test the length and that your result is a subsequence of both inputs.
- Return an empty string when there is no common character. Do not modify the inputs.
- Target: Θ(mn) time and Θ(mn) space for the table.

### Examples and checks

| a | b | One LCS | What to check |
| --- | --- | --- | --- |
| `"ABCB"` | `"BDCAB"` | `"BCB"` | Your Part A table |
| `"ABCBDAB"` | `"BDCABA"` | `"BCBA"` | Several valid answers (also "BDAB") |
| `"ACE"` | `"ABCDE"` | `"ACE"` | One input is a subsequence of the other |
| `"ABC"` | `"DEF"` | `""` | No common character |
| `""` | `"ABC"` | `""` | Empty input |

### Function interface (Java)

```java
public static String longestCommonSubsequence(String a, String b)
```

**Explain your answer:** why does the traceback return a common subsequence of maximum length? Give time and space bounds. Computing only the length needs just two rows of the table; why is that not enough to return the string itself?

<details><summary>Hint 1 · Two index systems</summary>

Table indices are prefix lengths: row i means the first i characters. The last character of that prefix is `a[i − 1]` in 0-based string indexing. Most LCS bugs are off-by-one errors between these two systems.

</details>

<details><summary>Hint 2 · Walk the table backwards</summary>

Start at (m, n). If `a[i − 1] == b[j − 1]`, record the character and move diagonally to (i − 1, j − 1). Otherwise move to whichever of (i − 1, j) and (i, j − 1) holds the larger value; on a tie, either works. Stop when i or j reaches 0. You collect characters from the end, so reverse at the end or fill an output buffer from the back.

</details>

Edit and run the starter definition in the next cell, then run its self-check.

In [ ]:
public class Solution {
    public static String longestCommonSubsequence(String a, String b) {
        int m = a.length(), n = b.length();
        // L[i][j] = LCS length of a's first i and b's first j characters.
        int[][] L = new int[m + 1][n + 1];
        for (int i = 1; i <= m; i++) {
            for (int j = 1; j <= n; j++) {
                // TODO: fill L[i][j] with the key math function.
                // L uses prefix lengths; the last characters are a.charAt(i - 1) and b.charAt(j - 1).
            }
        }
        // TODO: trace back from (m, n) and return one LCS as a string.
        throw new UnsupportedOperationException();
    }
}


In [ ]:
// Re-run the Solution definition cell before running this check cell.
// These checks throw while the starter is still unchanged.
java.util.function.BiPredicate<String, String> isSubsequence = (s, t) -> {
    int k = 0;
    for (int i = 0; i < t.length() && k < s.length(); i++) if (s.charAt(k) == t.charAt(i)) k++;
    return k == s.length();
};
String[][] cases = {
    {"", "", "0"}, {"", "ABC", "0"}, {"ABC", "DEF", "0"}, {"a", "A", "0"},
    {"ABCB", "BDCAB", "3"}, {"ABCBDAB", "BDCABA", "4"}, {"AGGTAB", "GXTXAYB", "4"},
    {"ABC", "ABC", "3"}, {"ACE", "ABCDE", "3"}, {"AAAA", "AA", "2"}
};
for (String[] test : cases) {
    String result = Solution.longestCommonSubsequence(test[0], test[1]);
    if (result == null || result.length() != Integer.parseInt(test[2]))
        throw new AssertionError("Wrong LCS length for " + test[0] + ", " + test[1] + ": " + result);
    if (!isSubsequence.test(result, test[0]) || !isSubsequence.test(result, test[1]))
        throw new AssertionError("Not a common subsequence: " + result);
}
String large = Solution.longestCommonSubsequence("AB".repeat(500), "BA".repeat(500));
if (large == null || large.length() != 999 || !isSubsequence.test(large, "AB".repeat(500))
        || !isSubsequence.test(large, "BA".repeat(500)))
    throw new AssertionError("Check the 1,000 x 1,000 case");
System.out.println("Exercise 2 checks passed (10 cases + 1,000 x 1,000 input).");


### Your explanation — Exercise 2, Part B

- **Traceback correctness:** Why is the returned string a common subsequence of maximum length?
- **Complexity:** Time and space of the fill and the traceback.
- **Two rows:** Why is a two-row table enough for the length but not for the string?


## Discussion and exit check

Compare your exchange argument in Exercise 1 with the argument for why matching equal last characters is safe in Exercise 2. Both start from some optimal solution and change it, step by step, into the one your algorithm builds, without making it worse.

Diagnose this LCS bug: the code fills the table with `if a[i] == b[j]` while i and j are table indices running from 1 to m and 1 to n. What goes wrong on the first row, and what goes wrong at `i = m`?

**Exit check:** greedy is optimal for `1 || Lmax` but not for coin change with `{1, 3, 4}`. In one or two sentences: what does the EDD proof rely on that has no counterpart for these coins?

Before leaving, keep your own copy of the notebook or code, and write down one remaining question for the TA.

### Your reflection

**LCS indexing bug:** What goes wrong with `a[i] == b[j]`, and how do you fix it?

**Exit check:** What does the EDD proof rely on that has no counterpart for coins {1, 3, 4}?


## Extra · Huffman coding

This section is provided by the TA for students who want to go further. It is not covered in lectures or on the final exam, and the core exercises do not depend on it. It applies the greedy and exchange-argument ideas from this session to a classic data-compression algorithm.

### Why we need it

Text is stored as bits. A fixed-length code gives every symbol the same number of bits: 6 distinct symbols need ⌈$\log_{2}$ 6⌉ = 3 bits each. Real data is skewed, though: in English text, `e` is far more common than `z`. A *variable-length* code gives frequent symbols short codewords and rare symbols long ones, which reduces the total number of bits.

Variable lengths raise a decoding problem. With a = 0, b = 1, c = 01, the bits `01` could mean "ab" or "c". A **prefix-free code** avoids this: no codeword is a prefix of another, so a decoder can read bits left to right and output a symbol as soon as the bits read so far match a codeword.

A prefix-free binary code is equivalent to a binary tree with the symbols at its leaves: going left adds a `0`, going right adds a `1`, and a symbol's codeword is the path from the root to its leaf. Leaves never lie on another leaf's path, which is exactly the prefix-free property. If symbol c appears f(c) times, the encoded length is `B(T) = Σ f(c) · depth(c)`. Huffman's algorithm finds a tree that minimizes B(T).

### How it works: merge the two rarest

Start with each symbol as a one-node tree weighted by its frequency. Repeatedly remove the two lightest trees, make them the children of a new node whose weight is their sum, and put the new tree back. After n − 1 merges a single tree remains. A min-priority queue (binary heap) finds the two lightest trees quickly.

```pseudocode
HUFFMAN(freq[0..n−1])                    // leaves are nodes 0..n−1
    if n = 1: return ["0"]
    Q = min-priority queue of nodes 0..n−1, keyed by weight = freq
    for node = n to 2n − 2:              // n − 1 merges
        x = EXTRACT-MIN(Q)
        y = EXTRACT-MIN(Q)
        left[node] = x;  right[node] = y
        weight[node] = weight[x] + weight[y]
        INSERT(Q, node)
    // node 2n − 2 is the root; every child was created before its parent
    code[2n − 2] = ""
    for node = 2n − 2 down to n:
        code[left[node]]  = code[node] + "0"
        code[right[node]] = code[node] + "1"
    return code[0..n−1]
```

### Worked example

A 100-character file uses six symbols with these frequencies. Each step merges the two lightest trees; the tree removed first becomes the left (0) child.

| Step | Merge (left + right) | New weight | Trees remaining |
| --- | --- | --- | --- |
| 0 | — | — | a:45, b:13, c:12, d:16, e:9, f:5 |
| 1 | f:5 + e:9 | 14 | a:45, b:13, c:12, d:16, (fe):14 |
| 2 | c:12 + b:13 | 25 | a:45, d:16, (fe):14, (cb):25 |
| 3 | (fe):14 + d:16 | 30 | a:45, (cb):25, (fed):30 |
| 4 | (cb):25 + (fed):30 | 55 | a:45, (cbfed):55 |
| 5 | a:45 + (cbfed):55 | 100 | root |

| Symbol | a | b | c | d | e | f |
| --- | --- | --- | --- | --- | --- | --- |
| Frequency | 45 | 13 | 12 | 16 | 9 | 5 |
| Huffman code | `0` | `101` | `100` | `111` | `1101` | `1100` |
| Fixed 3-bit code | `000` | `001` | `010` | `011` | `100` | `101` |

Huffman: 45·1 + 13·3 + 12·3 + 16·3 + 9·4 + 5·4 = 224 bits, compared with 300 bits for the fixed code (about 25% smaller). The total also equals the sum of the merged weights, 14 + 25 + 30 + 55 + 100 = 224: each merge adds one bit to every symbol beneath it.

### Why the greedy choice is safe (sketch)

1. **Exchange step.** In some optimal tree, the two least frequent symbols x and y are sibling leaves at the greatest depth. Take any optimal tree and swap x and y with the two deepest siblings. A rarer symbol moves down and a more frequent one moves up, so B(T) does not increase. This is the same idea as the swap in Exercise 1.
2. **Reduction step.** Merging x and y into one symbol z with f(z) = f(x) + f(y) gives a smaller instance, and B(T) = B(T′) + f(x) + f(y). An optimal tree for the smaller instance therefore gives an optimal tree for the original one.

**Cost:** with a binary heap, building the initial heap takes O(n) time, and each of the n − 1 merges does two extract-min operations and one insert in O(log n) time, so O(n log n) total. Writing the codes takes time proportional to their total length.

## Extra · Build Huffman codes

**Optional coding.** Given the frequency `freq[i]` of each symbol i, return its codeword `code[i]`, a string over `'0'` and `'1'`. The codes must be prefix-free and minimize Σ freq[i] · length(code[i]).

- 1 ≤ n ≤ 1,000 symbols; each frequency is between 1 and 1,000,000, so all merged weights fit in a 32-bit signed integer. The typed-language starters store weights in 64-bit integers for extra headroom.
- With a single symbol, return `["0"]`; a codeword must not be empty.
- Ties make the code non-unique. The checks test prefix-freeness and the optimal total cost, not exact strings. Do not modify `freq`.
- Use your language's priority queue: `heapq` (Python), `PriorityQueue` (Java, C#), or `std::priority_queue` (C++). The C and JavaScript starters include a small `MinHeap` helper, so you can focus on Huffman itself.

### Function interface (Java)

```java
public static String[] huffmanCodes(int[] freq)
```

**Going further (optional):** write a decoder that walks the tree bit by bit. What changes if you find the two minimum weights by scanning an array instead of using a heap? If the frequencies arrive already sorted, two plain queues (one for leaves, one for merged trees) give an O(n) algorithm. Why do merged weights come out in nondecreasing order?

<details><summary>Hint 1 · Store the tree in arrays</summary>

Number the leaves 0..n−1 and the merged nodes n..2n−2 in creation order. Then you only need `left[]`, `right[]`, and `weight[]` arrays; no node class is required. The last node created, 2n − 2, is the root.

</details>

<details><summary>Hint 2 · Codes without recursion</summary>

A parent is always created after its children. Visit nodes from 2n − 2 down to n: by the time you reach a node, its own code is already known, so give its left child that code + "0" and its right child that code + "1". (In C, compute each node's depth this way first, then fill each leaf's string from the back by following parent links.)

</details>

Edit and run the starter definition in the next cell, then run its self-check.

In [ ]:
import java.util.PriorityQueue;

public class Solution {
    public static String[] huffmanCodes(int[] freq) {
        int n = freq.length;
        if (n == 1) return new String[] {"0"};
        // Nodes 0..n-1 are leaves (symbols); nodes n..2n-2 are merged internal nodes.
        long[] weight = new long[2 * n - 1];
        int[] left = new int[2 * n - 1], right = new int[2 * n - 1];
        PriorityQueue<Integer> heap = new PriorityQueue<>((x, y) -> Long.compare(weight[x], weight[y]));
        for (int i = 0; i < n; i++) {
            weight[i] = freq[i];
            heap.add(i);
        }
        // TODO: for node = n .. 2n-2, poll the two lightest trees, record them as
        // this node's children, set its weight, and add it back.
        // TODO: assign codes from the root (node 2n-2) down: left adds "0", right adds "1".
        // Return the codes of nodes 0..n-1.
        throw new UnsupportedOperationException();
    }
}


In [ ]:
import java.util.Arrays;
import java.util.HashMap;
import java.util.Map;

// Re-run the Solution definition cell before running this check cell.
// These checks throw while the starter is still unchanged.
// Expected costs are the optimal totals sum(freq[i] * code[i].length()).
int[] nines = new int[9];
Arrays.fill(nines, 3);
int[][] freqs = {{7}, {5, 5}, {1, 1, 1, 1}, {45, 13, 12, 16, 9, 5},
                 {1, 2, 4, 8, 16}, {1000000, 1, 1}, nines, {2, 3, 7, 1, 1}};
long[] costs = {7, 10, 8, 224, 56, 1000004, 87, 27};
for (int t = 0; t < freqs.length; t++) {
    int[] freq = freqs[t], before = freq.clone();
    String[] codes = Solution.huffmanCodes(freq);
    if (codes == null || codes.length != freq.length) throw new AssertionError("Return one code per symbol");
    if (!Arrays.equals(freq, before)) throw new AssertionError("Do not modify freq");
    long total = 0;
    for (int k = 0; k < codes.length; k++) {
        if (codes[k] == null || !codes[k].matches("[01]+")) throw new AssertionError("Bad code: " + codes[k]);
        total += (long) freq[k] * codes[k].length();
    }
    String[] ordered = codes.clone();
    Arrays.sort(ordered);
    for (int k = 1; k < ordered.length; k++)
        if (ordered[k].startsWith(ordered[k - 1])) throw new AssertionError("Not prefix-free: " + Arrays.toString(codes));
    if (total != costs[t]) throw new AssertionError(Arrays.toString(freq) + ": cost " + total + ", optimal " + costs[t]);
}

// Round trip: encode a message with your codes, then decode it bit by bit.
String[] codes = Solution.huffmanCodes(new int[] {45, 13, 12, 16, 9, 5});
Map<String, Character> symbolOf = new HashMap<>();
for (int k = 0; k < codes.length; k++) symbolOf.put(codes[k], "abcdef".charAt(k));
String message = "faceadbead";
StringBuilder bits = new StringBuilder();
for (char ch : message.toCharArray()) bits.append(codes["abcdef".indexOf(ch)]);
StringBuilder decoded = new StringBuilder();
String current = "";
for (char bit : bits.toString().toCharArray()) {
    current += bit;
    if (symbolOf.containsKey(current)) {
        decoded.append(symbolOf.get(current));
        current = "";
    }
}
if (!decoded.toString().equals(message) || !current.isEmpty()) throw new AssertionError("Round trip failed");
System.out.println("Huffman checks passed (8 cases + round trip: " + message.length() + " symbols in "
        + bits.length() + " bits, vs " + 3 * message.length() + " fixed).");


## References and related material

- T. H. Cormen, C. E. Leiserson, R. L. Rivest, and C. Stein, *Introduction to Algorithms*, 4th ed. (MIT Press, 2022): Chapter 14 (Dynamic Programming: matrix-chain multiplication, LCS) and Chapter 15 (Greedy Algorithms: activity selection, Huffman codes).
- [D. A. Huffman, “A Method for the Construction of Minimum-Redundancy Codes,” Proceedings of the IRE, 1952](https://doi.org/10.1109/JRPROC.1952.273898) — the original paper, for the curious.
- [Recitation 2 · counting recursive work](https://harryc919.github.io/TA-CS301/recitation2.html#divide-conquer) — compare plain recursion with DP tables.
- [Recitation 0 · notebook environment and kernels](https://harryc919.github.io/TA-CS301/recitation0.html)